In [ ]:
## Demonstrating the Tensor Contiguity Issue

# Create a sample tensor (e.g., padded image tile)
original_tensor = torch.randn(3, 256, 256)  # [C, H, W]
print(f"Original tensor shape: {original_tensor.shape}")
print(f"Original tensor is_contiguous(): {original_tensor.is_contiguous()}")
print()

# Simulate padding (which can create non-contiguous tensors)
padded_tensor = torch.nn.functional.pad(original_tensor, (0, 32, 0, 32), value=0)
print(f"Padded tensor shape: {padded_tensor.shape}")
print(f"Padded tensor is_contiguous(): {padded_tensor.is_contiguous()}")
print()

# Attempting to view a non-contiguous tensor can cause issues
# Let's show what happens
try:
    # Some model operations might try to reshape the tensor
    viewed = padded_tensor.view(-1)  # Flatten
    print("✓ View succeeded (this may work in some cases)")
except RuntimeError as e:
    print(f"✗ View failed with error: {str(e)[:80]}...")
print()

# The fix: use .contiguous()
contiguous_tensor = torch.nn.functional.pad(original_tensor, (0, 32, 0, 32), value=0).contiguous()
print(f"Contiguous tensor is_contiguous(): {contiguous_tensor.is_contiguous()}")
try:
    viewed = contiguous_tensor.view(-1)
    print(f"✓ View succeeded! Flattened shape: {viewed.shape}")
except RuntimeError as e:
    print(f"✗ View failed: {e}")


## Section 1: Understanding the RuntimeError Issue

### The Problem
When using `torch.nn.functional.pad()` on tensors, the resulting tensors may have a non-contiguous memory layout. This occurs because padding can rearrange how data is stored in memory.

When models try to call `.view()` or `.reshape()` on these padded tensors, PyTorch throws:
```
RuntimeError: view size is not compatible with input tensor's size and stride 
(at least one dimension spans across two contiguous subspaces)
```

### The Solution
Call `.contiguous()` after padding to ensure the tensor has contiguous memory layout before passing it to models that expect contiguous tensors.

In [ ]:
import os
import tempfile
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from lightning import LightningDataModule
import logging

# Configure logging to see informative messages
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ PyTorch version:", torch.__version__)
print("✓ CUDA available:", torch.cuda.is_available())

# TilingDataModuleWrapper Tutorial

This notebook demonstrates how to use the TilingDataModuleWrapper to handle large images with consistent padding behavior across training and inference. It also shows how to fix tensor contiguity issues that can cause RuntimeErrors during finetuning.

## Key Topics
1. Understanding tensor contiguity issues in PyTorch
2. Fixing 'view size is not compatible' errors
3. Using TilingDataModuleWrapper for data tiling
4. Running finetune tests with multiple backbones
5. Inference with prediction stitching